In [1]:
# ============================================================
# BUILD DIM_DRIVERS — LOCAL GOLD
# ============================================================

import os
from pyspark.sql import functions as F

In [2]:
try:
    SILVER_ROOT
except NameError:
    import nbformat
    from nbconvert import PythonExporter
    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:09:29 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:09:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:09:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:09:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/08 23:09:30 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/08 23:09:30 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/08 23:09:30 

✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [3]:
silver_drivers = f"{SILVER_ROOT}/drivers/drivers_silver.parquet"
ref_nat_region = f"{GOLD_ROOT}/ref_nationality_region/ref_nationality_region.parquet"
gold_folder = f"{GOLD_ROOT}/dim_drivers"
gold_file = f"{gold_folder}/dim_drivers.parquet"
os.makedirs(gold_folder, exist_ok=True)

In [4]:
drivers_df = spark.read.parquet(silver_drivers)
ref_df = spark.read.parquet(ref_nat_region)

In [5]:
dim_drivers_df = (
    drivers_df.alias("d")
        .join(
            ref_df.alias("r"),
            F.col("d.nationality") == F.col("r.nationality"),
            "left"
        )
        .select(
            F.col("d.driver_id"),
            F.col("d.driver_name"),
            F.col("d.date_of_birth"),
            F.col("d.nationality"),
            F.col("r.region").alias("nationality_region")
        )
)

In [6]:
(
    dim_drivers_df
        .write
        .mode("overwrite")
        .parquet(gold_file)
)

print(f"✔ dim_drivers written to: {gold_file}")
spark.read.parquet(gold_file).show(10, truncate=False)

✔ dim_drivers written to: /Users/manoharazalki/F1-ANALYTICS/data/gold/dim_drivers/dim_drivers.parquet
+----------+-----------------+-------------+-----------+------------------+
|driver_id |driver_name      |date_of_birth|nationality|nationality_region|
+----------+-----------------+-------------+-----------+------------------+
|Cannoc    |John Cannon      |1933-06-21   |Canadian   |North America     |
|Changy    |Alain De Changy  |1922-02-05   |Belgian    |Europe            |
|abate     |Carlo Abate      |1932-07-10   |Italian    |Europe            |
|abecassis |George Abecassis |1913-03-21   |British    |Europe            |
|acheson   |Kenny Acheson    |1957-11-27   |British    |Europe            |
|adamich   |Andrea De Adamich|1941-10-03   |Italian    |Europe            |
|adams     |Philippe Adams   |1969-11-19   |Belgian    |Europe            |
|ader      |Walt Ader        |1913-12-15   |American   |North America     |
|adolff    |Kurt Adolff      |1921-11-05   |German     |Europe